# Lab 5 — Checkpoints with LangGraph (EP08 · framework lab)

In Lab 2 we wrote the state, the checkpoints, the resume and the rollback ourselves — this lab does **the same six-step task** with LangGraph.
In EP05 Lab 5 we used `InMemorySaver` to pause for approval — this time we use **`SqliteSaver`**, which writes to a **file**, so it survives the program closing

| Lab 2 (by hand) | LangGraph |
|---|---|
| `TaskState` (dataclass) | `State` (TypedDict) + a reducer |
| `run_task` walks the plan, skipping finished steps | a graph of connected nodes · the graph knows which node is next |
| `save_state` after every step (`os.replace`) | the checkpointer saves after every node (to SQLite) |
| `load_state(CKPT)` then `run_task` | `invoke(None, config)` with the same `thread_id` |
| `rollback(state, 2)` | `get_state_history` + `update_state` → a new branch |
| `version` | not provided — the state's schema is still ours to look after |

The papers and the network drop are **simulated data for teaching**. The model's answers and the checkpoint counts come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.6, §18.9 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep08-lab5-langgraph-durable.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) |
| Packages | `langgraph`, `langgraph-checkpoint`, `langgraph-checkpoint-sqlite` (pinned versions), `openai`, `python-dotenv` |

How to add the key, and why we chose this model and provider, are in **Lab 1**

In [ ]:
%pip install -q "langgraph==1.2.16" "langgraph-checkpoint==4.3.1" "langgraph-checkpoint-sqlite==3.1.3" "openai>=1.50" python-dotenv

In [ ]:
import operator, os, sqlite3
from typing import Annotated, TypedDict
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["makora"], "allow_fallbacks": False}}
USAGE = {"calls": 0}

def chat(prompt):
    USAGE["calls"] += 1
    reply = client.chat.completions.create(model=MODEL, max_tokens=250, extra_body=EXTRA,
                                           messages=[{"role": "user", "content": prompt}])
    return reply.choices[0].message.content

PAPERS = {
    "P1": "Informer (2021): ProbSparse self-attention makes long-sequence time-series forecasting faster.",
    "P2": "PatchTST (2023): splits each series into patches of 16 steps; each channel is handled on its own.",
    "P3": "TimesFM (2024): a decoder-only foundation model that forecasts new series zero-shot.",
}

## 1. The state and six nodes

`results` uses the **reducer** `operator.or_` — each node returns only its own output, and LangGraph merges it into the existing dict (`a | b`).
`audience` = who the report is for, which we'll change when we roll back

In [ ]:
# ✏️ TODO 1 (▶ video 1:19): tell LangGraph to merge the results dicts with operator.or_
class State(TypedDict):
    audience: str
    results: ___   # each node's output is merged in

In [ ]:
STEPS = [
    ["note_P1", "In one sentence, what is the key idea of this paper? {P1}"],
    ["note_P2", "In one sentence, what is the key idea of this paper? {P2}"],
    ["note_P3", "In one sentence, what is the key idea of this paper? {P3}"],
    ["compare", "Compare three papers in 3 short bullet points, using these notes:\n{note_P1}\n{note_P2}\n{note_P3}"],
    ["title",   "Write one title for a 2-page report for {audience}, built on this comparison. Reply with the title only.\n{compare}"],
    ["report",  "Write the report's opening paragraph for {audience}, under 80 words.\nTitle: {title}\nComparison:\n{compare}"],
]
NETWORK = {"drop_at": None}

In [ ]:
def make_node(name, template):
    def node(state):
        if NETWORK["drop_at"] == name:
            NETWORK["drop_at"] = None
            raise ConnectionError(f"network dropped at step '{name}'")
        print(f"  ✓ {name}")
        prompt = template.format(**PAPERS, audience=state["audience"], **state["results"])
        return {"results": {name: chat(prompt)}}
    return node

## 2. A straight-line graph + `SqliteSaver`

The checkpointer saves the state **after every node** into the file `report.db` — each task is kept apart by its `thread_id`

In [ ]:
# ✏️ TODO 2 (▶ video 2:15): draw an edge from the previous node to this one
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

builder = StateGraph(State)
previous = START
for name, template in STEPS:
    builder.add_node(name, make_node(name, template))
    ___
    previous = name
builder.add_edge(previous, END)

# ✏️ TODO 3 (▶ video 2:32): compile with a SqliteSaver checkpointer connected to the DB file

DB = "report.db"
if os.path.exists(DB):
    os.remove(DB)
graph = builder.compile(checkpointer=___)
print(" → ".join(graph.get_graph().nodes))

## 3. Crash midway, then resume from the file

In [ ]:
config = {"configurable": {"thread_id": "report-1"}}
NETWORK["drop_at"] = "compare"
try:
    graph.invoke({"audience": "researchers", "results": {}}, config)
except ConnectionError as e:
    print("💥", e)

In [ ]:
del graph   # as if the program had closed
graph = builder.compile(checkpointer=SqliteSaver(sqlite3.connect(DB, check_same_thread=False)))

snapshot = graph.get_state(config)
print("next:", snapshot.next, "· done:", list(snapshot.values["results"]))

In [ ]:
# ✏️ TODO 4 (▶ video 3:32): resume from the latest checkpoint (input = None)
result = ___
print(f"model calls in total: {USAGE['calls']}\n")
print(result["results"]["title"])

## 4. What the framework adds: every checkpoint, and branching

Lab 2 kept only the latest checkpoint, and its rollback **deleted** the old outputs.
LangGraph keeps **every** checkpoint — go back to any of them, change the state, and branch off a **new branch** while the old one stays

In [ ]:
history = list(graph.get_state_history(config))
print(f"{len(history)} checkpoints, newest first:")
for h in history:
    print("  next:", h.next, "· done:", len(h.values.get("results", {})))

In [ ]:
# ✏️ TODO 5 (▶ video 4:28): branch off from before_title, changing the audience to first-year students
USAGE["calls"] = 0
before_title = next(h for h in history if h.next == ("title",))
fork = ___
new = graph.invoke(None, fork)
print(f"model calls: {USAGE['calls']}\n\n{new['results']['report']}")

In [ ]:
print(len(list(graph.get_state_history(config))), "checkpoints now")
print("old branch:", history[0].values["results"]["report"][:90], "…")

## Summary

| What we did | Lab 2 (by hand) | Lab 5 (LangGraph) |
|---|---|---|
| State | dataclass + `version` | TypedDict + reducer (the schema's version is still ours to manage) |
| Saving | our own `save_state` after every step | the checkpointer saves after every node into a SQLite file |
| Crash at step 4, resume | 6 calls | 6 calls · no load/skip code of our own |
| Go back 2 steps | old outputs deleted · 2 calls | a new branch · 2 calls · the old branch stays (8 → 11 checkpoints) |

A framework saves code and adds what's hard to write yourself (every checkpoint, branching) — but we need to know what it saves, when, and where

**Exercises**
1. **A real restart:** after the network drop in part 3, restart the kernel (Runtime → Restart), run the cells that define `State`, `STEPS`, `make_node` and build `builder` again, then `invoke(None, config)` — does it resume?
2. Use a new `thread_id` (`"report-2"`) for the same task, and count how many checkpoints each thread's `get_state_history` has

**Next — the EP08 Recap**